In [1]:
import json
import math
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

DATA_DIR = "/kaggle/input/ml-challenge-udhgam-2"

def load_jsonl(path):
    rows = []
    with open(path, "r") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

train_data = load_jsonl(f"{DATA_DIR}/train.jsonl")
test_data  = load_jsonl(f"{DATA_DIR}/test.jsonl")
print(f"Train samples: {len(train_data)}")
print(f"Test samples : {len(test_data)}")

print("\nSample train row:")
train_data[0]
lengths = [len(x["input_ids"]) for x in train_data]
labels  = [x["label"] for x in train_data]

print("Sequence length stats:")
print(pd.Series(lengths).describe())

print("\nLabel stats:")
print(pd.Series(labels).describe())

Using device: cuda
Train samples: 79806
Test samples : 19952

Sample train row:
Sequence length stats:
count    79806.000000
mean        90.075633
std         86.562309
min          3.000000
25%         20.000000
50%         55.000000
75%        132.000000
max        256.000000
dtype: float64

Label stats:
count    79806.000000
mean         0.523029
std          0.325152
min          0.000000
25%          0.204038
50%          0.566820
75%          0.838201
max          1.000000
dtype: float64


In [2]:
# =========================
# Global setup
# =========================
from collections import Counter, defaultdict
import numpy as np
import zlib

MAX_LEN = 256

GLOBAL_CNT = Counter(t for row in train_data for t in row["input_ids"])
TOTAL_TOKENS = sum(GLOBAL_CNT.values())

TOP_COMMON_TOKENS = set(t for t, _ in GLOBAL_CNT.most_common(50))
COMMON_TOKENS = TOP_COMMON_TOKENS

TOP_K = 200
TOP_K_TOKENS = [t for t, _ in GLOBAL_CNT.most_common(TOP_K)]

RARE_Q = [0.001, 0.01, 0.1]

FEATURE_DIM = 34 + TOP_K + 8  # keep in sync


def extract_features(row):
    tokens = [t for t, m in zip(row["input_ids"], row["attention_mask"]) if m == 1]
    L = len(tokens)

    if L == 0:
        return np.zeros(FEATURE_DIM, dtype=np.float32)

    feats = []

    # =====================================================
    # 1. Length & density
    # =====================================================
    uniq = len(set(tokens))
    feats += [
        L,
        np.log1p(L),
        uniq / L,
        L / max(uniq, 1),
        1 - L / MAX_LEN
    ]

    # =====================================================
    # 2. Global entropy & repetition
    # =====================================================
    cnt = Counter(tokens)
    freqs = np.array(list(cnt.values())) / L

    feats += [
        -np.sum(freqs * np.log(freqs + 1e-8)) / np.log(L + 1e-8),
        np.sum(freqs ** 2),
        np.max(freqs),
        np.sum(np.sort(freqs)[-3:])
    ]

    feats.append(
        np.mean([tokens[i] == tokens[i+1] for i in range(L-1)]) if L > 1 else 0
    )

    # =====================================================
    # 3. Positional identity (hashed)
    # =====================================================
    def hash_id(t, mod=50):
        return (t % mod) / mod

    lead = tokens[:3] if L >= 3 else tokens
    tail = tokens[-3:] if L >= 3 else tokens

    feats += [hash_id(t) for t in lead]
    feats += [hash_id(t) for t in tail]

    # =====================================================
    # 4. Token recurrence distance
    # =====================================================
    pos = defaultdict(list)
    for i, t in enumerate(tokens):
        pos[t].append(i)

    gaps = [d for p in pos.values() if len(p) > 1 for d in np.diff(p)]
    feats += [np.mean(gaps), np.std(gaps), np.max(gaps)] if gaps else [0, 0, 0]

    # =====================================================
    # 5. Structural anchor variance
    # =====================================================
    anchor_std = []
    for tok, _ in cnt.most_common(5):
        p = pos[tok]
        if len(p) > 2:
            anchor_std.append(np.std(np.diff(p)))
    feats.append(np.mean(anchor_std) if anchor_std else 0)

    # =====================================================
    # 6. N-gram repetition
    # =====================================================
    for n in [2, 3]:
        ngrams = [tuple(tokens[i:i+n]) for i in range(L-n+1)]
        c = Counter(ngrams)
        feats += [
            max(c.values()) / len(ngrams),
            sum(v > 1 for v in c.values()) / len(c)
        ] if c else [0, 0]

    # =====================================================
    # 7. Bigram entropy
    # =====================================================
    if L > 2:
        bigrams = [(tokens[i], tokens[i+1]) for i in range(L-1)]
        bc = Counter(bigrams)
        p = np.array(list(bc.values())) / len(bigrams)
        feats.append(-np.sum(p * np.log(p + 1e-8)))
    else:
        feats.append(0)

    # =====================================================
    # 8. Rarity profile
    # =====================================================
    global_freqs = np.array([GLOBAL_CNT[t] / TOTAL_TOKENS for t in tokens])

    feats += [
        np.mean(global_freqs < RARE_Q[0]),
        np.mean((global_freqs >= RARE_Q[0]) & (global_freqs < RARE_Q[1])),
        np.mean((global_freqs >= RARE_Q[1]) & (global_freqs < RARE_Q[2])),
        np.mean(global_freqs >= RARE_Q[2]),
        np.mean(global_freqs),
        np.std(global_freqs),
        np.min(global_freqs)
    ]

    # =====================================================
    # 9. Compression ratio
    # =====================================================
    raw = np.array(tokens, dtype=np.uint32).tobytes()
    feats.append(len(zlib.compress(raw)) / max(len(raw), 1))

    # =====================================================
    # 10. Top-K identity
    # =====================================================
    token_set = set(tokens)
    feats += [1.0 if t in token_set else 0.0 for t in TOP_K_TOKENS]

    # =====================================================
    # 11. Delimiter detection
    # =====================================================
    feats.append(sum((t in COMMON_TOKENS) and (cnt[t] <= 3) for t in cnt))

    # =====================================================
    # 12. 🔥 WINDOW ENTROPY (PHASE SHIFT SIGNAL)
    # =====================================================
    def window_entropy(seg):
        if len(seg) < 2:
            return 0
        c = Counter(seg)
        p = np.array(list(c.values())) / len(seg)
        return -np.sum(p * np.log(p + 1e-8)) / np.log(len(seg) + 1e-8)

    q1 = L // 4
    q2 = L // 2
    q3 = 3 * L // 4

    e1 = window_entropy(tokens[:q1])
    e2 = window_entropy(tokens[q1:q2])
    e3 = window_entropy(tokens[q2:q3])
    e4 = window_entropy(tokens[q3:])

    feats += [
        e1, e2, e3, e4,
        e2 - e1,
        e3 - e2,
        e4 - e3,
        max(e1, e2, e3, e4) - min(e1, e2, e3, e4)
    ]

    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)
    return np.array(feats, dtype=np.float32)

MAX_LEN = 256
BATCH_SIZE = 64
KERNEL_SIZES = [3, 5, 7]

EMBED_DIM = 128
NUM_FILTERS = 128
DROPOUT = 0.2
def compute_features(input_ids, attention_mask):
    """
    input_ids: List[int]
    attention_mask: List[int]
    Returns: np.array of shape (4,)
    """
    # Keep only real tokens
    tokens = [t for t, m in zip(input_ids, attention_mask) if m == 1]

    length = len(tokens)

    if length == 0:
        return np.zeros(4, dtype=np.float32)

    # 1. Normalized length
    norm_len = length / MAX_LEN

    # 2. Unique token ratio
    unique_ratio = len(set(tokens)) / length

    # 3. Token entropy
    counts = {}
    for t in tokens:
        counts[t] = counts.get(t, 0) + 1

    probs = np.array(list(counts.values()), dtype=np.float32) / length
    entropy = -np.sum(probs * np.log(probs + 1e-8))

    # Normalize entropy by log(length)
    entropy = entropy / np.log(length + 1e-8)

    # 4. Repetition ratio (adjacent)
    repeats = sum(tokens[i] == tokens[i+1] for i in range(length - 1))
    repetition_ratio = repeats / max(1, length - 1)

    return np.array(
        [norm_len, unique_ratio, entropy, repetition_ratio],
        dtype=np.float32
    )

all_ids = []

for row in train_data:
    all_ids.extend(row["input_ids"])

for row in test_data:
    all_ids.extend(row["input_ids"])

max_token_id = max(all_ids)
VOCAB_SIZE = max_token_id + 1

print("Max token id:", max_token_id)
print("Vocab size:", VOCAB_SIZE)

class PromptDataset(Dataset):
    def __init__(self, rows, indices, is_test=False):
        self.rows = rows
        self.indices = indices
        self.is_test = is_test

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        row = self.rows[self.indices[i]]

        input_ids = row["input_ids"][:MAX_LEN]
        attention = row["attention_mask"][:MAX_LEN]

        feats = compute_features(input_ids, attention)

        pad = MAX_LEN - len(input_ids)
        if pad > 0:
            input_ids += [0] * pad
            attention += [0] * pad

        item = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),

            "attention": torch.tensor(attention, dtype=torch.float),
            "features": torch.tensor(feats, dtype=torch.float),
        }

        if not self.is_test:
            item["label"] = torch.tensor(row["label"], dtype=torch.float)

        return item


Max token id: 50367
Vocab size: 50368


In [3]:
class AttnPoolCNNPromptModel(nn.Module):
    def __init__(self, vocab_size, num_features=4):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            EMBED_DIM,
            padding_idx=0
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(EMBED_DIM, NUM_FILTERS, k, padding=k // 2)
            for k in KERNEL_SIZES
        ])

        # attention scorer (shared across kernels)
        self.attn_fc = nn.Linear(NUM_FILTERS, 1)

        self.dropout = nn.Dropout(DROPOUT)

        cnn_dim = len(KERNEL_SIZES) * NUM_FILTERS * 3  # attn + mean + max
        total_dim = cnn_dim + num_features

        self.regressor = nn.Sequential(
            nn.Linear(total_dim, 256),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(256, 1)
        )

    def attention_pool(self, x, mask):
        """
        x: (B, C, L)
        mask: (B, 1, L)
        """
        # (B, L, C)
        x_t = x.transpose(1, 2)

        scores = self.attn_fc(torch.tanh(x_t)).squeeze(-1)
        scores = scores.masked_fill(mask.squeeze(1) == 0, -1e9)

        attn = torch.softmax(scores, dim=1)
        pooled = torch.sum(x_t * attn.unsqueeze(-1), dim=1)

        return pooled

    def forward(self, input_ids, attention_mask, features):
        x = self.embedding(input_ids)          # (B, L, D)
        x = x.transpose(1, 2)                  # (B, D, L)
        mask = attention_mask.unsqueeze(1)

        pooled = []

        for conv in self.convs:
            c = torch.relu(conv(x)) * mask

            attn_pool = self.attention_pool(c, mask)
            mean_pool = c.sum(dim=2) / (mask.sum(dim=2) + 1e-6)
            max_pool = c.max(dim=2).values

            pooled.extend([attn_pool, mean_pool, max_pool])

        cnn_features = torch.cat(pooled, dim=1)
        cnn_features = self.dropout(cnn_features)

        all_features = torch.cat([cnn_features, features], dim=1)

        out = self.regressor(all_features).squeeze(1)
        return torch.sigmoid(out)
class AttnGRUPromptModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_features=4):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # scalar attention per timestep
        self.attn_fc = nn.Linear(2 * hidden_dim, 1)

        self.dropout = nn.Dropout(0.3)

        total_dim = 2 * hidden_dim + num_features

        self.regressor = nn.Sequential(
            nn.Linear(total_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, input_ids, attention_mask, features):
        x = self.embedding(input_ids)                # (B, L, D)

        lengths = attention_mask.sum(dim=1).long().cpu()

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths, batch_first=True, enforce_sorted=False
        )

        packed_out, _ = self.gru(packed)

        out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=MAX_LEN
        )                                            # (B, L, 2H)

        mask = attention_mask.unsqueeze(-1)

        # ---------- attention pooling ----------
        scores = self.attn_fc(torch.tanh(out)).squeeze(-1)  # (B, L)
        scores = scores.masked_fill(attention_mask == 0, -1e9)

        attn = torch.softmax(scores, dim=1)
        attn_pool = torch.sum(out * attn.unsqueeze(-1), dim=1)

        # ---------- stability fallback ----------
        mean_pool = (out * mask).sum(dim=1) / (mask.sum(dim=1) + 1e-6)

        rep = 0.7 * attn_pool + 0.3 * mean_pool
        rep = self.dropout(rep)

        out = self.regressor(torch.cat([rep, features], dim=1)).squeeze(1)
        return torch.sigmoid(out)

In [4]:
class SegmentedGRUPromptModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_features=4):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )

        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.2)

        # 3 segments × 2*hidden_dim
        total_dim = 3 * (2 * hidden_dim) + num_features

        self.regressor = nn.Sequential(
            nn.Linear(total_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, input_ids, attention_mask, features):
        x = self.embedding(input_ids)  # (B, L, D)

        lengths = attention_mask.sum(dim=1).long().cpu()

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths, batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.gru(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=MAX_LEN
        )  # (B, L, 2H)

        mask = attention_mask.unsqueeze(-1)
        out = out * mask

        reps = []
        for i, L in enumerate(lengths):
            L = int(L.item())
            if L < 3:
                pooled = out[i, :L].mean(dim=0)
                reps.append(torch.cat([pooled, pooled, pooled]))
            else:
                s1 = out[i, :L//3].mean(dim=0)
                s2 = out[i, L//3:2*L//3].mean(dim=0)
                s3 = out[i, 2*L//3:L].mean(dim=0)
                reps.append(torch.cat([s1, s2, s3]))

        gru_features = torch.stack(reps)
        gru_features = self.dropout(gru_features)

        all_features = torch.cat([gru_features, features], dim=1)
        out = self.regressor(all_features).squeeze(1)

        return torch.sigmoid(out)


class CNNPromptModel(nn.Module):
    def __init__(self, vocab_size=50000, num_features=4):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=EMBED_DIM,
            padding_idx=0
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=EMBED_DIM,
                out_channels=NUM_FILTERS,
                kernel_size=k,
                padding=k // 2
            )
            for k in KERNEL_SIZES
        ])

        self.dropout = nn.Dropout(DROPOUT)

        cnn_out_dim = len(KERNEL_SIZES) * NUM_FILTERS * 2
        total_dim = cnn_out_dim + num_features

        self.regressor = nn.Sequential(
            nn.Linear(total_dim, 128),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(128, 1)
        )

    def forward(self, input_ids, attention_mask, features):
        x = self.embedding(input_ids)
        x = x.transpose(1, 2)

        pooled_outputs = []

        for conv in self.convs:
            c = torch.relu(conv(x))
            mask = attention_mask.unsqueeze(1)
            c = c * mask

            max_pool = torch.max(c, dim=2).values
            mean_pool = torch.sum(c, dim=2) / (mask.sum(dim=2) + 1e-6)

            pooled_outputs.append(max_pool)
            pooled_outputs.append(mean_pool)

        cnn_features = torch.cat(pooled_outputs, dim=1)
        cnn_features = self.dropout(cnn_features)

        all_features = torch.cat([cnn_features, features], dim=1)

        out = self.regressor(all_features).squeeze(1)
        return torch.sigmoid(out)
class GRUPromptModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_features=4):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )

        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.2)

        # GRU output: 2 * hidden_dim (bi)
        total_dim = 2 * hidden_dim + num_features

        self.regressor = nn.Sequential(
            nn.Linear(total_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, input_ids, attention_mask, features):
        x = self.embedding(input_ids)          # (B, L, D)

        lengths = attention_mask.sum(dim=1).long().cpu()

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths, batch_first=True, enforce_sorted=False
        )

        packed_out, _ = self.gru(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=MAX_LEN
        )                                       # (B, L, 2H)

        mask = attention_mask.unsqueeze(-1)

        out = out * mask

        mean_pool = out.sum(dim=1) / (mask.sum(dim=1) + 1e-6)
        max_pool = out.max(dim=1).values

        gru_features = 0.5 * (mean_pool + max_pool)
        gru_features = self.dropout(gru_features)

        all_features = torch.cat([gru_features, features], dim=1)

        out = self.regressor(all_features).squeeze(1)
        return torch.sigmoid(out)
# ===============================
# CNN kernel sizes (must match training)
# ===============================

KERNEL_SIZES = [3, 5, 7]

class DilatedCNNPromptModel(nn.Module):
    def __init__(self, vocab_size, num_features=4):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=EMBED_DIM,
            padding_idx=0
        )

        # Dilated convolutions
        self.convs = nn.ModuleList([
            nn.Conv1d(EMBED_DIM, NUM_FILTERS, kernel_size=3, dilation=1, padding=1),
            nn.Conv1d(EMBED_DIM, NUM_FILTERS, kernel_size=3, dilation=2, padding=2),
            nn.Conv1d(EMBED_DIM, NUM_FILTERS, kernel_size=3, dilation=4, padding=4),
            nn.Conv1d(EMBED_DIM, NUM_FILTERS, kernel_size=3, dilation=8, padding=8),
        ])

        self.dropout = nn.Dropout(DROPOUT)

        cnn_out_dim = len(self.convs) * NUM_FILTERS * 2
        total_dim = cnn_out_dim + num_features

        self.regressor = nn.Sequential(
            nn.Linear(total_dim, 128),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(128, 1)
        )

    def forward(self, input_ids, attention_mask, features):
        x = self.embedding(input_ids)        # (B, L, D)
        x = x.transpose(1, 2)                # (B, D, L)

        pooled = []
        mask = attention_mask.unsqueeze(1)

        for conv in self.convs:
            c = torch.relu(conv(x))
            c = c * mask

            max_pool = c.max(dim=2).values
            mean_pool = c.sum(dim=2) / (mask.sum(dim=2) + 1e-6)

            pooled.append(max_pool)
            pooled.append(mean_pool)

        cnn_features = torch.cat(pooled, dim=1)
        cnn_features = self.dropout(cnn_features)

        all_features = torch.cat([cnn_features, features], dim=1)
        out = self.regressor(all_features).squeeze(1)

        return torch.sigmoid(out)


In [5]:
y = np.array([r["label"] for r in train_data])
N = len(train_data)

In [6]:
import os, random, json
import numpy as np
import torch

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
from sklearn.model_selection import KFold
import pickle

N_FOLDS = 5
FOLD_PATH = "folds.pkl"

if os.path.exists(FOLD_PATH):
    print("Loading existing folds")
    with open(FOLD_PATH, "rb") as f:
        folds = pickle.load(f)
else:
    print("Creating new folds")
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    folds = list(kf.split(np.arange(N)))
    with open(FOLD_PATH, "wb") as f:
        pickle.dump(folds, f)

# sanity
for i, (tr, va) in enumerate(folds):
    print(f"Fold {i}: train={len(tr)}, val={len(va)}")


Device: cuda
Creating new folds
Fold 0: train=63844, val=15962
Fold 1: train=63845, val=15961
Fold 2: train=63845, val=15961
Fold 3: train=63845, val=15961
Fold 4: train=63845, val=15961


In [7]:
def train_one_fold(
    model,
    train_idx,
    val_idx,
    fold,
    model_name,
    epochs=15,          # ← increased
    lr=2e-4,
    batch_size=64,
    patience=3,         # ← early stopping
):
    train_ds = PromptDataset(train_data, train_idx)
    val_ds   = PromptDataset(train_data, val_idx)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, drop_last=False
    )
    val_loader   = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False
    )

    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    best_mae = 1e9
    best_state = None
    bad_epochs = 0

    for ep in range(epochs):
        # =====================
        # TRAIN
        # =====================
        model.train()
        train_preds, train_ys = [], []

        for b in train_loader:
            opt.zero_grad()

            preds = model(
                b["input_ids"].to(DEVICE),
                b["attention"].to(DEVICE),
                b["features"].to(DEVICE),
            )

            y = b["label"].to(DEVICE)
            loss = torch.mean(torch.abs(preds - y))

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            train_preds.append(preds.detach().cpu().numpy())
            train_ys.append(y.cpu().numpy())

        train_mae = mean_absolute_error(
            np.concatenate(train_ys),
            np.concatenate(train_preds),
        )

        # =====================
        # VALIDATION
        # =====================
        model.eval()
        val_preds, val_ys = [], []

        with torch.no_grad():
            for b in val_loader:
                preds = model(
                    b["input_ids"].to(DEVICE),
                    b["attention"].to(DEVICE),
                    b["features"].to(DEVICE),
                )

                val_preds.append(preds.cpu().numpy())
                val_ys.append(b["label"].numpy())

        val_mae = mean_absolute_error(
            np.concatenate(val_ys),
            np.concatenate(val_preds),
        )

        print(
            f"[{model_name} | Fold {fold}] "
            f"Epoch {ep+1}/{epochs} | "
            f"Train MAE {train_mae:.5f} | Val MAE {val_mae:.5f}"
        )

        # =====================
        # SAVE BEST + EARLY STOP
        # =====================
        if val_mae < best_mae:
            best_mae = val_mae
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            bad_epochs = 0

            torch.save(
                best_state,
                f"checkpoints/{model_name}_fold{fold}.pt"
            )
            print("🔥 Saved best weights")
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print("⏹ Early stopping triggered")
                break

    # restore best weights
    model.load_state_dict(best_state)
    return model, best_mae


In [8]:
os.makedirs("checkpoints", exist_ok=True)


In [9]:
MODEL_ZOO = {
    "cnn": CNNPromptModel,
    "cnn_seed2": CNNPromptModel,
    "bigru": GRUPromptModel,
    "bigru_seed2": GRUPromptModel,
    "seg_gru": SegmentedGRUPromptModel,
    "attn_gru": AttnGRUPromptModel,
    "attn_cnn": AttnPoolCNNPromptModel,
    "dilated_cnn": DilatedCNNPromptModel,
}
oof_preds = {name: np.zeros(N) for name in MODEL_ZOO}
test_preds = {name: np.zeros(len(test_data)) for name in MODEL_ZOO}

test_idx = np.arange(len(test_data))
test_ds = PromptDataset(test_data, test_idx, is_test=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

os.makedirs("checkpoints", exist_ok=True)

for name, ModelClass in MODEL_ZOO.items():
    print(f"\n===== MODEL {name} =====")
    fold_test_preds = []

    for f, (tr_idx, va_idx) in enumerate(folds):
        print(f"Fold {f}")

        seed_everything(SEED + f + hash(name) % 1000)

        model = ModelClass(VOCAB_SIZE)

        model, best_mae = train_one_fold(
            model,
            train_idx=tr_idx,
            val_idx=va_idx,
            fold=f,
            model_name=name,
            epochs=15,
            lr=2e-4,
            batch_size=64,
        )

        # ---------- OOF ----------
        val_ds = PromptDataset(train_data, va_idx)
        val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

        model.eval()
        with torch.no_grad():
            preds = []
            for b in val_loader:
                p = model(
                    b["input_ids"].to(DEVICE),
                    b["attention"].to(DEVICE),
                    b["features"].to(DEVICE),
                )
                preds.append(p.cpu().numpy())

        oof_preds[name][va_idx] = np.concatenate(preds)

        # ---------- TEST ----------
        tp = []
        with torch.no_grad():
            for b in test_loader:
                p = model(
                    b["input_ids"].to(DEVICE),
                    b["attention"].to(DEVICE),
                    b["features"].to(DEVICE),
                )
                tp.append(p.cpu().numpy())

        fold_test_preds.append(np.concatenate(tp))

    test_preds[name] = np.mean(fold_test_preds, axis=0)



===== MODEL cnn =====
Fold 0
[cnn | Fold 0] Epoch 1/15 | Train MAE 0.20531 | Val MAE 0.18339
🔥 Saved best weights
[cnn | Fold 0] Epoch 2/15 | Train MAE 0.17798 | Val MAE 0.17839
🔥 Saved best weights
[cnn | Fold 0] Epoch 3/15 | Train MAE 0.16622 | Val MAE 0.17365
🔥 Saved best weights
[cnn | Fold 0] Epoch 4/15 | Train MAE 0.15746 | Val MAE 0.17010
🔥 Saved best weights
[cnn | Fold 0] Epoch 5/15 | Train MAE 0.14985 | Val MAE 0.16536
🔥 Saved best weights
[cnn | Fold 0] Epoch 6/15 | Train MAE 0.14350 | Val MAE 0.16584
[cnn | Fold 0] Epoch 7/15 | Train MAE 0.13876 | Val MAE 0.16617
[cnn | Fold 0] Epoch 8/15 | Train MAE 0.13392 | Val MAE 0.16340
🔥 Saved best weights
[cnn | Fold 0] Epoch 9/15 | Train MAE 0.12995 | Val MAE 0.16487
[cnn | Fold 0] Epoch 10/15 | Train MAE 0.12542 | Val MAE 0.16318
🔥 Saved best weights
[cnn | Fold 0] Epoch 11/15 | Train MAE 0.12232 | Val MAE 0.16426
[cnn | Fold 0] Epoch 12/15 | Train MAE 0.11892 | Val MAE 0.16326
[cnn | Fold 0] Epoch 13/15 | Train MAE 0.11548 | Val